# Olist Marketing Funnel & Lead Conversion Analytics
**Colab notebook: data check → cleaning → SQL → KPIs → Excel export**

**Business challenge:** Olist's sales team turns Marketing Qualified Leads (MQLs) into sellers. Where are we losing leads, and which lead sources convert?

**Measurable objectives**
1. Measure the MQL → Closed Deal conversion rate (overall, by lead origin, by month cohort).
2. Measure sales-cycle length (days from first contact to won date).
3. Profile what the closed deals look like (lead type, segment, behaviour profile, SDR, SR).
4. Export clean tables to Excel for the dashboard.

**How to run:** Runtime → Run all. When the upload box appears, select BOTH CSV files. At the end an Excel file downloads automatically.


## 1. Upload & load

In [ ]:
import os, io
import pandas as pd, numpy as np
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)

try:
    from google.colab import files
    uploaded = files.upload()          # choose both CSV files
except ImportError:
    print('Not running in Colab - reading files from DATA_DIR')

DATA_DIR = os.environ.get('DATA_DIR', '.')
mql = pd.read_csv(os.path.join(DATA_DIR, 'olist_marketing_qualified_leads_dataset.csv'))
cd  = pd.read_csv(os.path.join(DATA_DIR, 'olist_closed_deals_dataset.csv'))
print('MQL rows/cols   :', mql.shape)
print('Closed deals    :', cd.shape)

Saving olist_closed_deals_dataset.csv to olist_closed_deals_dataset (1).csv
Saving olist_marketing_qualified_leads_dataset.csv to olist_marketing_qualified_leads_dataset (2).csv
MQL rows/cols   : (8000, 4)
Closed deals    : (842, 14)


## 2. Does the data match the project? (column check)

In [ ]:
expected_mql = ['mql_id','first_contact_date','landing_page_id','origin']
expected_cd  = ['mql_id','seller_id','sdr_id','sr_id','won_date','business_segment','lead_type',
                'lead_behaviour_profile','has_company','has_gtin','average_stock','business_type',
                'declared_product_catalog_size','declared_monthly_revenue']
print('MQL columns missing   :', [c for c in expected_mql if c not in mql.columns] or 'None - all present')
print('Closed columns missing:', [c for c in expected_cd  if c not in cd.columns]  or 'None - all present')
print('Row count check -> MQLs:', len(mql), '| Closed deals:', len(cd))
print('Every closed deal exists in MQL table?', cd['mql_id'].isin(mql['mql_id']).all())

MQL columns missing   : None - all present
Closed columns missing: None - all present
Row count check -> MQLs: 8000 | Closed deals: 842
Every closed deal exists in MQL table? True


## 3. Data dictionary (plain language)

In [ ]:
dictionary = pd.DataFrame([
 ['MQL','mql_id','Unique ID of each marketing-qualified lead (join key)'],
 ['MQL','first_contact_date','Date the lead first filled the form / was contacted (start of sales clock)'],
 ['MQL','landing_page_id','Landing page the lead signed up on'],
 ['MQL','origin','Marketing channel the lead came from (organic_search, paid_search, social, ...)'],
 ['Closed','mql_id','Links the deal back to its MQL'],
 ['Closed','seller_id','Seller account created after the deal was won'],
 ['Closed','sdr_id','Sales Development Rep - qualifies the lead / first conversations'],
 ['Closed','sr_id','Sales Representative - closes the deal'],
 ['Closed','won_date','Date-time the deal was won (end of sales clock)'],
 ['Closed','business_segment','Product category the seller will sell (home_decor, pet, ...)'],
 ['Closed','lead_type','Kind of seller (online_medium, industry, offline, ...)'],
 ['Closed','lead_behaviour_profile','Behaviour persona (cat, eagle, wolf, shark; some have two)'],
 ['Closed','has_company','Seller has a registered company? (questionnaire answer)'],
 ['Closed','has_gtin','Products have barcodes (GTIN)? (questionnaire answer)'],
 ['Closed','average_stock','Declared stock range'],
 ['Closed','business_type','reseller / manufacturer / other'],
 ['Closed','declared_product_catalog_size','Number of products the seller declared'],
 ['Closed','declared_monthly_revenue','Revenue the seller declared per month (BRL)'],
], columns=['Table','Column','Meaning'])
dictionary

,Table,Column,Meaning
0,MQL,mql_id,Unique ID of each marketing-qualified lead (jo...
1,MQL,first_contact_date,Date the lead first filled the form / was cont...
2,MQL,landing_page_id,Landing page the lead signed up on
3,MQL,origin,Marketing channel the lead came from (organic_...
4,Closed,mql_id,Links the deal back to its MQL
5,Closed,seller_id,Seller account created after the deal was won
6,Closed,sdr_id,Sales Development Rep - qualifies the lead / f...
7,Closed,sr_id,Sales Representative - closes the deal
8,Closed,won_date,Date-time the deal was won (end of sales clock)
9,Closed,business_segment,Product category the seller will sell (home_de...


## 4. Data-quality check

In [ ]:
def profile(df, name):
    out = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'missing_n': df.isna().sum(),
        'missing_%': (df.isna().mean()*100).round(2),
        'unique': df.nunique()})
    print(f'--- {name} ---'); display(out)
profile(mql, 'MQL'); profile(cd, 'Closed deals')

dq = pd.DataFrame([
 ['Duplicate mql_id (MQL)',       int(mql['mql_id'].duplicated().sum())],
 ['Duplicate mql_id (Closed)',    int(cd['mql_id'].duplicated().sum())],
 ['Duplicate full rows (MQL)',    int(mql.duplicated().sum())],
 ['Duplicate full rows (Closed)', int(cd.duplicated().sum())],
 ['Closed deals with no matching MQL', int((~cd['mql_id'].isin(mql['mql_id'])).sum())],
 ['MQLs with missing origin',     int(mql['origin'].isna().sum())],
], columns=['Check','Count'])
dq

--- MQL ---


,dtype,missing_n,missing_%,unique
mql_id,object,0,0.00,8000
first_contact_date,object,0,0.00,336
landing_page_id,object,0,0.00,495
origin,object,60,0.75,10


--- Closed deals ---


,dtype,missing_n,missing_%,unique
mql_id,object,0,0.00,842
seller_id,object,0,0.00,842
sdr_id,object,0,0.00,32
sr_id,object,0,0.00,22
won_date,object,0,0.00,824
business_segment,object,1,0.12,33
lead_type,object,6,0.71,8
lead_behaviour_profile,object,177,21.02,9
has_company,object,779,92.52,2
has_gtin,object,778,92.40,2


,Check,Count
0,Duplicate mql_id (MQL),0
1,Duplicate mql_id (Closed),0
2,Duplicate full rows (MQL),0
3,Duplicate full rows (Closed),0
4,Closed deals with no matching MQL,0
5,MQLs with missing origin,60


### Missing values: what does NULL mean here?
| Column | Missing | Meaning / decision |
|---|---|---|
| `origin` (MQL) | ~0.75% | Channel not recorded. Different from the existing `'unknown'` value, so we label it `not_provided` and keep both. |
| `lead_behaviour_profile` | ~21% | Profile never assigned. Report as `Not Provided`; never guess a persona. |
| `has_company`, `has_gtin`, `average_stock`, `declared_product_catalog_size` | ~92% | Questionnaire fields filled for only a small group of deals. NULL = *not answered*, **not** "No" / 0. Keep as NULL; use only as supporting info. |
| `declared_monthly_revenue` | 0% missing, but ~95% are 0 | A 0 most likely means *nothing declared*. One extreme value (50,000,000). **Not used in KPIs**. |
| `lead_type`, `business_segment`, `business_type` | ≤1.2% | Very few; labelled `Not Provided`. |

## 5. Cleaning (nothing is deleted; every decision is documented)

In [ ]:
mql_c = mql.copy(); cd_c = cd.copy()

# text hygiene
for col in ['origin']:
    mql_c[col] = mql_c[col].str.strip().str.lower()
for col in ['business_segment','lead_type','lead_behaviour_profile','business_type','average_stock']:
    cd_c[col] = cd_c[col].str.strip().str.lower()

# dates: first_contact_date is a DATE, won_date is a DATE-TIME -> compare on date only
mql_c['first_contact_date'] = pd.to_datetime(mql_c['first_contact_date'], errors='coerce')
cd_c['won_date'] = pd.to_datetime(cd_c['won_date'], errors='coerce')
cd_c['won_day']  = cd_c['won_date'].dt.normalize()
print('Unparseable dates -> MQL:', mql_c['first_contact_date'].isna().sum(), '| Closed:', cd_c['won_date'].isna().sum())

# labels for reporting only (raw NULLs stay NULL in the original columns)
mql_c['origin_label'] = mql_c['origin'].fillna('not_provided')
for col in ['business_segment','lead_type','lead_behaviour_profile','business_type']:
    cd_c[col + '_label'] = cd_c[col].fillna('Not Provided')

# flag multi-value behaviour profiles such as 'cat, wolf'
cd_c['behaviour_is_combo'] = cd_c['lead_behaviour_profile'].fillna('').str.contains(',')
print('Combined behaviour profiles:', int(cd_c['behaviour_is_combo'].sum()))

Unparseable dates -> MQL: 0 | Closed: 0
Combined behaviour profiles: 16


## 6. Master analytical table (MQL LEFT JOIN closed deals)

In [ ]:
master = mql_c.merge(cd_c, on='mql_id', how='left', validate='one_to_one')

# conversion status: converted if a closed-deal record exists
master['conversion_status'] = np.where(master['seller_id'].notna(), 'Converted', 'Not Converted')
master['converted_flag']    = (master['conversion_status'] == 'Converted').astype(int)

# days to close = won date - first contact date (converted leads only)
master['days_to_close_raw'] = (master['won_day'] - master['first_contact_date']).dt.days
neg = master['days_to_close_raw'] < 0
print('Deals with won date BEFORE first contact (data error):', int(neg.sum()))
master['days_to_close'] = master['days_to_close_raw'].where(~neg)      # negative -> NULL, row kept

master['first_contact_month'] = master['first_contact_date'].dt.to_period('M').astype(str)
master['won_month'] = master['won_day'].dt.to_period('M').astype(str).replace('NaT', np.nan)

print(master.shape); master.head()

Deals with won date BEFORE first contact (data error): 1
(8000, 30)


,mql_id,first_contact_date,landing_page_id,origin,origin_label,seller_id,sdr_id,sr_id,won_date,business_segment,lead_type,lead_behaviour_profile,has_company,has_gtin,average_stock,business_type,declared_product_catalog_size,declared_monthly_revenue,won_day,business_segment_label,lead_type_label,lead_behaviour_profile_label,business_type_label,behaviour_is_combo,conversion_status,converted_flag,days_to_close_raw,days_to_close,first_contact_month,won_month
0,dac32acd4db4c29c230538b72f8dd87d,2018-02-01,88740e65d5d6b056e0cda098e1ea6313,social,social,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,Not Converted,0,NaN,NaN,2018-02,NaN
1,8c18d1de7f67e60dbd64e3c07d7e9d5d,2017-10-20,007f9098284a86ee80ddeb25d53e0af8,paid_search,paid_search,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,Not Converted,0,NaN,NaN,2017-10,NaN
2,b4bc852d233dfefc5131f593b538befa,2018-03-22,a7982125ff7aa3b2054c6e44f9d28522,organic_search,organic_search,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,Not Converted,0,NaN,NaN,2018-03,NaN
3,6be030b81c75970747525b843c1ef4f8,2018-01-22,d45d558f0daeecf3cccdffe3c59684aa,email,email,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,Not Converted,0,NaN,NaN,2018-01,NaN
4,5420aad7fec3549a85876ba1c529bd84,2018-02-21,b48ec5f3b04e9068441002a19df93c6c,organic_search,organic_search,2c43fb513632d29b3b58df74816f1b06,a8387c01a09e99ce014107505b92388c,4ef15afb4b2723d8f3d81e51ec7afefe,2018-02-26 19:58:54,pet,online_medium,cat,NaN,NaN,NaN,reseller,NaN,0.0,2018-02-26,pet,online_medium,cat,reseller,False,Converted,1,5.0,5.0,2018-02,2018-02


## 7. SQL analysis (runs inside Colab using SQLite)
The queries are standard SQL. Where MySQL differs, the MySQL version is in a comment (e.g. `DATEDIFF`, `DATE_FORMAT`).

In [ ]:
import sqlite3
con = sqlite3.connect(':memory:')
m2 = mql_c.copy();  m2['first_contact_date'] = m2['first_contact_date'].dt.strftime('%Y-%m-%d')
c2 = cd_c.copy();   c2['won_date'] = c2['won_date'].dt.strftime('%Y-%m-%d'); c2['won_day'] = c2['won_day'].dt.strftime('%Y-%m-%d')
m2.drop(columns=['origin_label']).to_sql('mql', con, index=False)
c2[['mql_id','seller_id','sdr_id','sr_id','won_date','won_day','business_segment','lead_type',
    'lead_behaviour_profile','business_type']].to_sql('closed_deals', con, index=False)

def q(sql): return pd.read_sql_query(sql, con)

# Q1-Q4  Core KPIs
q('''
SELECT COUNT(*) AS total_mqls,
       COUNT(c.mql_id) AS closed_deals,
       COUNT(*) - COUNT(c.mql_id) AS non_converted,
       ROUND(100.0 * COUNT(c.mql_id) / COUNT(*), 2) AS conversion_rate_pct
FROM mql m
LEFT JOIN closed_deals c ON m.mql_id = c.mql_id;
''')

,total_mqls,closed_deals,non_converted,conversion_rate_pct
0,8000,842,7158,10.53


In [ ]:
# Q5  Analytical table with conversion_status (the CASE WHEN from the project brief)
q('''
SELECT m.mql_id, m.first_contact_date, COALESCE(m.origin,'not_provided') AS origin,
       c.won_day, c.lead_type, c.business_segment, c.sdr_id, c.sr_id, c.seller_id,
       CASE WHEN c.mql_id IS NOT NULL THEN 'Converted' ELSE 'Not Converted' END AS conversion_status
FROM mql m
LEFT JOIN closed_deals c ON m.mql_id = c.mql_id
LIMIT 10;
''')

,mql_id,first_contact_date,origin,won_day,lead_type,business_segment,sdr_id,sr_id,seller_id,conversion_status
0,dac32acd4db4c29c230538b72f8dd87d,2018-02-01,social,None,None,None,None,None,None,Not Converted
1,8c18d1de7f67e60dbd64e3c07d7e9d5d,2017-10-20,paid_search,None,None,None,None,None,None,Not Converted
2,b4bc852d233dfefc5131f593b538befa,2018-03-22,organic_search,None,None,None,None,None,None,Not Converted
3,6be030b81c75970747525b843c1ef4f8,2018-01-22,email,None,None,None,None,None,None,Not Converted
4,5420aad7fec3549a85876ba1c529bd84,2018-02-21,organic_search,2018-02-26,online_medium,pet,a8387c01a09e99ce014107505b92388c,4ef15afb4b2723d8f3d81e51ec7afefe,2c43fb513632d29b3b58df74816f1b06,Converted
5,28bdfd5f057764b54c38770f95c69f2f,2018-01-14,organic_search,None,None,None,None,None,None,Not Converted
6,126a0d10becbaafcb2e72ce6848cf32c,2018-05-15,email,None,None,None,None,None,None,Not Converted
7,f76136f54d14a3345951f25b7932366b,2018-05-24,email,None,None,None,None,None,None,Not Converted
8,2f838cade4a6012a6cb1016d1d8d95ed,2017-11-10,organic_search,None,None,None,None,None,None,Not Converted
9,7281942387a1a0c3f72a50a8b0bb0920,2017-12-25,social,None,None,None,None,None,None,Not Converted


In [ ]:
# Q6  Conversion rate by origin  (valid: every MQL has an origin)
q('''
SELECT COALESCE(m.origin,'not_provided') AS origin,
       COUNT(*) AS total_mqls,
       COUNT(c.mql_id) AS closed_deals,
       COUNT(*) - COUNT(c.mql_id) AS non_converted,
       ROUND(100.0 * COUNT(c.mql_id) / COUNT(*), 2) AS conversion_rate_pct
FROM mql m LEFT JOIN closed_deals c ON m.mql_id = c.mql_id
GROUP BY COALESCE(m.origin,'not_provided')
ORDER BY total_mqls DESC;
''')

,origin,total_mqls,closed_deals,non_converted,conversion_rate_pct
0,organic_search,2296,271,2025,11.80
1,paid_search,1586,195,1391,12.30
2,social,1350,75,1275,5.56
3,unknown,1099,179,920,16.29
4,direct_traffic,499,56,443,11.22
5,email,493,15,478,3.04
6,referral,284,24,260,8.45
7,other,150,4,146,2.67
8,display,118,6,112,5.08
9,other_publicities,65,3,62,4.62


In [ ]:
# Q7  Monthly COHORT conversion (grouped by the month the lead first made contact)
# MySQL: DATE_FORMAT(m.first_contact_date,'%Y-%m')
q('''
SELECT SUBSTR(m.first_contact_date,1,7) AS contact_month,
       COUNT(*) AS mqls,
       COUNT(c.mql_id) AS closed_from_cohort,
       ROUND(100.0 * COUNT(c.mql_id) / COUNT(*), 2) AS cohort_conversion_pct
FROM mql m LEFT JOIN closed_deals c ON m.mql_id = c.mql_id
GROUP BY SUBSTR(m.first_contact_date,1,7)
ORDER BY contact_month;
''')

,contact_month,mqls,closed_from_cohort,cohort_conversion_pct
0,2017-06,4,0,0.00
1,2017-07,239,2,0.84
2,2017-08,386,9,2.33
3,2017-09,312,7,2.24
4,2017-10,416,14,3.37
5,2017-11,445,18,4.04
6,2017-12,200,11,5.50
7,2018-01,1141,152,13.32
8,2018-02,1028,149,14.49
9,2018-03,1174,167,14.22


In [ ]:
# Q8  Closed deals by SDR (closed-deal table only -> volume and share, NOT conversion rate)
q('''
SELECT sdr_id, COUNT(*) AS closed_deals,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM closed_deals), 2) AS share_of_closed_pct
FROM closed_deals GROUP BY sdr_id ORDER BY closed_deals DESC LIMIT 10;
''')

,sdr_id,closed_deals,share_of_closed_pct
0,4b339f9567d060bcea4f5136b9f5949e,140,16.63
1,068066e24f0c643eb1d089c7dd20cd73,81,9.62
2,56bf83c4bb35763a51c2baab501b4c67,74,8.79
3,9d12ef1a7eca3ec58c545c678af7869c,66,7.84
4,a8387c01a09e99ce014107505b92388c,59,7.01
5,9e4d1098a3b0f5da39b0bc48f9876645,55,6.53
6,de63de0d10a6012430098db33c679b0b,53,6.29
7,370c9f455f93a9a96cbe9bea48e70033,51,6.06
8,f42a2bd194f7802ab052a815c8de65b7,42,4.99
9,09285259593c61296eef10c734121d5b,42,4.99


In [ ]:
# Q9  Days to close by lead type
# MySQL: DATEDIFF(c.won_day, m.first_contact_date)
q('''
SELECT COALESCE(c.lead_type,'Not Provided') AS lead_type,
       COUNT(*) AS deals,
       ROUND(AVG(julianday(c.won_day) - julianday(m.first_contact_date)),1) AS avg_days_to_close
FROM mql m JOIN closed_deals c ON m.mql_id = c.mql_id
WHERE julianday(c.won_day) - julianday(m.first_contact_date) >= 0
GROUP BY COALESCE(c.lead_type,'Not Provided')
ORDER BY deals DESC;
''')

,lead_type,deals,avg_days_to_close
0,online_medium,332,44.8
1,online_big,125,53.6
2,industry,123,46.5
3,offline,104,46.9
4,online_small,77,57.1
5,online_beginner,57,51.2
6,online_top,14,21.4
7,Not Provided,6,39.5
8,other,3,253.7


## 8. KPIs and funnel
**What the data supports:** MQL generated → Closed deal (→ seller created).
**Conceptual only (no records in the data):** Contacted, Consultancy. We do **not** invent counts for these.

In [ ]:
conv = master[master['converted_flag'] == 1]
valid_days = conv['days_to_close'].dropna()

kpi = pd.DataFrame([
 ['Total MQLs',                      len(master)],
 ['Total Closed Deals',              int(master['converted_flag'].sum())],
 ['Overall Conversion Rate (%)',     round(master['converted_flag'].mean()*100, 2)],
 ['Non-Converted Leads',             int((1-master['converted_flag']).sum())],
 ['Drop-off (%)',                    round((1-master['converted_flag'].mean())*100, 2)],
 ['Average Days to Close',           round(valid_days.mean(), 1)],
 ['Median Days to Close',            float(valid_days.median())],
 ['Min Days to Close',               int(valid_days.min())],
 ['Max Days to Close',               int(valid_days.max())],
 ['Unique SDRs',                     int(conv['sdr_id'].nunique())],
 ['Unique Sales Reps (SR)',          int(conv['sr_id'].nunique())],
 ['Deals excluded from days-to-close (negative date gap)', int(neg.sum())],
], columns=['KPI','Value'])

funnel = pd.DataFrame({
  'Stage': ['MQLs generated', 'Closed deals'],
  'Count': [len(master), int(master['converted_flag'].sum())]})
funnel['Stage_Conversion_%'] = [100.0, round(funnel['Count'][1]/funnel['Count'][0]*100, 2)]
funnel['Dropped_From_Previous'] = [0, funnel['Count'][0]-funnel['Count'][1]]
display(kpi); display(funnel)

,KPI,Value
0,Total MQLs,8000.00
1,Total Closed Deals,842.00
2,Overall Conversion Rate (%),10.52
3,Non-Converted Leads,7158.00
4,Drop-off (%),89.48
5,Average Days to Close,48.50
6,Median Days to Close,14.00
7,Min Days to Close,0.00
8,Max Days to Close,427.00
9,Unique SDRs,32.00


,Stage,Count,Stage_Conversion_%,Dropped_From_Previous
0,MQLs generated,8000,100.00,0
1,Closed deals,842,10.52,7158


## 9. Lead origin analysis (true conversion rate)
Look at **volume AND conversion together**. A small channel with a high rate may not be reliable (`n < 100` flagged).

In [ ]:
origin = (master.groupby('origin_label')
          .agg(Total_MQLs=('mql_id','count'), Closed_Deals=('converted_flag','sum')).reset_index()
          .rename(columns={'origin_label':'Origin'}))
origin['Non_Converted'] = origin['Total_MQLs'] - origin['Closed_Deals']
origin['Conversion_Rate_%'] = (origin['Closed_Deals']/origin['Total_MQLs']*100).round(2)
origin['Share_of_MQLs_%']   = (origin['Total_MQLs']/origin['Total_MQLs'].sum()*100).round(2)
origin['Share_of_Closed_%'] = (origin['Closed_Deals']/origin['Closed_Deals'].sum()*100).round(2)
origin['Sample_Flag'] = np.where(origin['Total_MQLs'] < 100, 'Small sample - interpret with care', 'OK')
origin = origin.sort_values('Total_MQLs', ascending=False).reset_index(drop=True)
origin

,Origin,Total_MQLs,Closed_Deals,Non_Converted,Conversion_Rate_%,Share_of_MQLs_%,Share_of_Closed_%,Sample_Flag
0,organic_search,2296,271,2025,11.80,28.70,32.19,OK
1,paid_search,1586,195,1391,12.30,19.82,23.16,OK
2,social,1350,75,1275,5.56,16.88,8.91,OK
3,unknown,1099,179,920,16.29,13.74,21.26,OK
4,direct_traffic,499,56,443,11.22,6.24,6.65,OK
5,email,493,15,478,3.04,6.16,1.78,OK
6,referral,284,24,260,8.45,3.55,2.85,OK
7,other,150,4,146,2.67,1.88,0.48,OK
8,display,118,6,112,5.08,1.48,0.71,OK
9,other_publicities,65,3,62,4.62,0.81,0.36,Small sample - interpret with care


## 10. Monthly cohort analysis

In [ ]:
cohort = (master.groupby('first_contact_month')
          .agg(MQLs=('mql_id','count'), Closed_From_Cohort=('converted_flag','sum')).reset_index()
          .rename(columns={'first_contact_month':'Month'}))
cohort['Cohort_Conversion_%'] = (cohort['Closed_From_Cohort']/cohort['MQLs']*100).round(2)

won_by_month = (conv.groupby('won_month').size().rename('Deals_Won_In_Month')
                .reset_index().rename(columns={'won_month':'Month'}))
monthly = cohort.merge(won_by_month, on='Month', how='outer').sort_values('Month').fillna(0)
for c in ['MQLs','Closed_From_Cohort','Deals_Won_In_Month']: monthly[c] = monthly[c].astype(int)
monthly

,Month,MQLs,Closed_From_Cohort,Cohort_Conversion_%,Deals_Won_In_Month
0,2017-06,4,0,0.00,0
1,2017-07,239,2,0.84,0
2,2017-08,386,9,2.33,0
3,2017-09,312,7,2.24,0
4,2017-10,416,14,3.37,0
5,2017-11,445,18,4.04,0
6,2017-12,200,11,5.50,3
7,2018-01,1141,152,13.32,73
8,2018-02,1028,149,14.49,113
9,2018-03,1174,167,14.22,147


## 11. Closed-deal profile tables (lead type, segment, behaviour)
Because non-converted leads have no lead type / segment / behaviour, these tables show **what closed deals look like** (counts, share, speed). They are **not** conversion rates.

In [ ]:
def profile_table(col, name, min_n=1):
    t = (conv.groupby(col)
         .agg(Closed_Deals=('mql_id','count'),
              Avg_Days_To_Close=('days_to_close','mean'),
              Median_Days_To_Close=('days_to_close','median')).reset_index()
         .rename(columns={col: name}))
    t['Share_of_Closed_%'] = (t['Closed_Deals']/t['Closed_Deals'].sum()*100).round(2)
    t['Avg_Days_To_Close'] = t['Avg_Days_To_Close'].round(1)
    t['Sample_Flag'] = np.where(t['Closed_Deals'] < 20, 'Small sample (<20)', 'OK')
    return t.sort_values('Closed_Deals', ascending=False).reset_index(drop=True)

lead_type_tbl = profile_table('lead_type_label', 'Lead_Type')
segment_tbl   = profile_table('business_segment_label', 'Business_Segment')
behaviour_tbl = profile_table('lead_behaviour_profile_label', 'Behaviour_Profile')
display(lead_type_tbl); display(behaviour_tbl); display(segment_tbl.head(10))

,Lead_Type,Closed_Deals,Avg_Days_To_Close,Median_Days_To_Close,Share_of_Closed_%,Sample_Flag
0,online_medium,332,44.8,12.0,39.43,OK
1,online_big,126,53.6,13.0,14.96,OK
2,industry,123,46.5,15.0,14.61,OK
3,offline,104,46.9,16.0,12.35,OK
4,online_small,77,57.1,17.0,9.14,OK
5,online_beginner,57,51.2,13.0,6.77,OK
6,online_top,14,21.4,6.0,1.66,Small sample (<20)
7,Not Provided,6,39.5,27.0,0.71,Small sample (<20)
8,other,3,253.7,260.0,0.36,Small sample (<20)


,Behaviour_Profile,Closed_Deals,Avg_Days_To_Close,Median_Days_To_Close,Share_of_Closed_%,Sample_Flag
0,cat,407,40.8,13.0,48.34,OK
1,Not Provided,177,39.4,15.0,21.02,OK
2,eagle,123,36.0,10.0,14.61,OK
3,wolf,95,83.9,36.0,11.28,OK
4,shark,24,75.4,13.5,2.85,OK
5,"cat, wolf",8,185.8,159.5,0.95,Small sample (<20)
6,"eagle, wolf",3,149.3,97.0,0.36,Small sample (<20)
7,"eagle, cat",3,199.0,203.0,0.36,Small sample (<20)
8,"shark, cat",1,181.0,181.0,0.12,Small sample (<20)
9,"shark, wolf",1,341.0,341.0,0.12,Small sample (<20)


,Business_Segment,Closed_Deals,Avg_Days_To_Close,Median_Days_To_Close,Share_of_Closed_%,Sample_Flag
0,home_decor,105,50.1,13.0,12.47,OK
1,health_beauty,93,35.5,12.0,11.05,OK
2,car_accessories,77,50.5,14.0,9.14,OK
3,household_utilities,71,52.8,12.0,8.43,OK
4,construction_tools_house_garden,69,60.7,11.0,8.19,OK
5,audio_video_electronics,64,63.2,22.0,7.60,OK
6,computers,34,55.0,13.0,4.04,OK
7,pet,30,29.1,14.5,3.56,OK
8,food_supplement,28,38.3,19.5,3.33,OK
9,food_drink,26,31.6,16.0,3.09,OK


## 12. Sales team (SDR and SR)

In [ ]:
def team_table(col, name):
    t = (conv.groupby(col)
         .agg(Closed_Deals=('mql_id','count'),
              Avg_Days_To_Close=('days_to_close','mean'),
              Median_Days_To_Close=('days_to_close','median')).reset_index()
         .rename(columns={col: name}).sort_values('Closed_Deals', ascending=False).reset_index(drop=True))
    t['Share_of_Closed_%'] = (t['Closed_Deals']/t['Closed_Deals'].sum()*100).round(2)
    t['Cumulative_Share_%'] = t['Share_of_Closed_%'].cumsum().round(2)
    t['Avg_Days_To_Close'] = t['Avg_Days_To_Close'].round(1)
    t['Rank'] = np.arange(1, len(t)+1)
    return t
sdr_tbl = team_table('sdr_id', 'SDR_ID'); sr_tbl = team_table('sr_id', 'SR_ID')
print('Top 5 SDRs hold', sdr_tbl['Share_of_Closed_%'].head(5).sum().round(1), '% of closed deals')
print('Top 5 SRs  hold', sr_tbl['Share_of_Closed_%'].head(5).sum().round(1), '% of closed deals')
display(sdr_tbl.head(10)); display(sr_tbl.head(10))

Top 5 SDRs hold 49.9 % of closed deals
Top 5 SRs  hold 49.4 % of closed deals


,SDR_ID,Closed_Deals,Avg_Days_To_Close,Median_Days_To_Close,Share_of_Closed_%,Cumulative_Share_%,Rank
0,4b339f9567d060bcea4f5136b9f5949e,140,33.6,13.0,16.63,16.63,1
1,068066e24f0c643eb1d089c7dd20cd73,81,31.2,12.0,9.62,26.25,2
2,56bf83c4bb35763a51c2baab501b4c67,74,27.1,11.0,8.79,35.04,3
3,9d12ef1a7eca3ec58c545c678af7869c,66,45.5,14.5,7.84,42.88,4
4,a8387c01a09e99ce014107505b92388c,59,26.0,10.0,7.01,49.89,5
5,9e4d1098a3b0f5da39b0bc48f9876645,55,18.3,11.0,6.53,56.42,6
6,de63de0d10a6012430098db33c679b0b,53,33.0,14.0,6.29,62.71,7
7,370c9f455f93a9a96cbe9bea48e70033,51,50.8,11.0,6.06,68.77,8
8,09285259593c61296eef10c734121d5b,42,36.7,9.0,4.99,73.76,9
9,f42a2bd194f7802ab052a815c8de65b7,42,78.3,17.5,4.99,78.75,10


,SR_ID,Closed_Deals,Avg_Days_To_Close,Median_Days_To_Close,Share_of_Closed_%,Cumulative_Share_%,Rank
0,4ef15afb4b2723d8f3d81e51ec7afefe,133,30.5,11.0,15.80,15.80,1
1,d3d1e91a157ea7f90548eef82f1955e3,82,46.6,10.5,9.74,25.54,2
2,6565aa9ce3178a5caf6171827af3a9ba,74,25.8,11.5,8.79,34.33,3
3,85fc447d336637ba1df43e793199fbc8,64,33.7,10.0,7.60,41.93,4
4,495d4e95a8cf8bbf8b432b612a2aa328,63,49.4,16.5,7.48,49.41,5
5,2695de1affa7750089c0455f8ce27021,59,48.4,16.0,7.01,56.42,6
6,fbf4aef3f6915dc0c3c97d6812522f6a,59,22.0,10.0,7.01,63.43,7
7,de63de0d10a6012430098db33c679b0b,53,61.7,15.0,6.29,69.72,8
8,9ae085775a198122c5586fa830ff7f2b,51,41.7,17.0,6.06,75.78,9
9,c638112b43f1d1b86dcabb0da720c901,36,22.4,10.0,4.28,80.06,10


> SDR/SR numbers here are **closed deals handled**, not conversion rate, because we do not know how many un-converted leads each person worked on. Do not rank people as "best" or "worst".

## 13. Time-to-close

In [ ]:
def ttc(col, name, min_n=10):
    t = (conv.dropna(subset=['days_to_close']).groupby(col)['days_to_close']
         .agg(Leads='count', Average_Days_To_Close='mean', Median_Days_To_Close='median',
              Min_Days='min', Max_Days='max').reset_index().rename(columns={col: name}))
    t['Average_Days_To_Close'] = t['Average_Days_To_Close'].round(1)
    t['Sample_Flag'] = np.where(t['Leads'] < min_n, f'Small sample (<{min_n})', 'OK')
    return t.sort_values('Leads', ascending=False).reset_index(drop=True)

ttc_lead  = ttc('lead_type_label', 'Lead_Type')
ttc_origin= ttc('origin_label', 'Origin')
ttc_seg   = ttc('business_segment_label', 'Business_Segment')

bins = pd.cut(valid_days, [-1,7,30,90,10_000], labels=['0-7 days','8-30 days','31-90 days','90+ days'])
ttc_bands = bins.value_counts().reindex(['0-7 days','8-30 days','31-90 days','90+ days']).rename_axis('Days_Band').reset_index(name='Closed_Deals')
ttc_bands['Share_%'] = (ttc_bands['Closed_Deals']/ttc_bands['Closed_Deals'].sum()*100).round(1)
display(ttc_lead); display(ttc_bands)
print('Mean is pulled up by a long tail; median is the better "typical" number.')

,Lead_Type,Leads,Average_Days_To_Close,Median_Days_To_Close,Min_Days,Max_Days,Sample_Flag
0,online_medium,332,44.8,12.0,0.0,341.0,OK
1,online_big,125,53.6,13.0,0.0,341.0,OK
2,industry,123,46.5,15.0,0.0,369.0,OK
3,offline,104,46.9,16.0,0.0,373.0,OK
4,online_small,77,57.1,17.0,0.0,427.0,OK
5,online_beginner,57,51.2,13.0,0.0,316.0,OK
6,online_top,14,21.4,6.0,0.0,130.0,OK
7,Not Provided,6,39.5,27.0,10.0,87.0,Small sample (<10)
8,other,3,253.7,260.0,195.0,306.0,Small sample (<10)


,Days_Band,Closed_Deals,Share_%
0,0-7 days,283,33.7
1,8-30 days,271,32.2
2,31-90 days,137,16.3
3,90+ days,150,17.8


Mean is pulled up by a long tail; median is the better "typical" number.


## 14. Export everything to ONE Excel workbook

In [ ]:
out = 'Olist_Funnel_Excel_Data.xlsx'
mql_export = mql_c.drop(columns=['origin_label'])
cd_export  = cd_c.drop(columns=[c for c in cd_c.columns if c.endswith('_label')] + ['behaviour_is_combo'])
master_export = master[['mql_id','first_contact_date','first_contact_month','origin_label','landing_page_id',
    'conversion_status','converted_flag','won_day','won_month','days_to_close','seller_id','sdr_id','sr_id',
    'lead_type_label','business_segment_label','lead_behaviour_profile_label','business_type_label']].rename(columns={
    'origin_label':'origin','lead_type_label':'lead_type','business_segment_label':'business_segment',
    'lead_behaviour_profile_label':'behaviour_profile','business_type_label':'business_type'})
master_export['first_contact_date'] = master_export['first_contact_date'].dt.date
master_export['won_day'] = pd.to_datetime(master_export['won_day']).dt.date

sheets = {
 'Master_Data': master_export,            # use this for PivotTables
 'MQL_Data': mql_export, 'Closed_Deals': cd_export,
 'Funnel_KPI': kpi, 'Funnel': funnel,
 'Origin_Analysis': origin, 'Monthly_Analysis': monthly,
 'Lead_Type_Profile': lead_type_tbl, 'Segment_Profile': segment_tbl, 'Behaviour_Profile': behaviour_tbl,
 'SDR_Analysis': sdr_tbl, 'SR_Analysis': sr_tbl,
 'Time_to_Close': ttc_lead, 'TTC_Origin': ttc_origin, 'TTC_Segment': ttc_seg, 'TTC_Bands': ttc_bands,
 'Data_Dictionary': dictionary, 'Data_Quality': dq,
}
with pd.ExcelWriter(out, engine='openpyxl', datetime_format='yyyy-mm-dd') as xw:
    for name, df in sheets.items():
        df.to_excel(xw, sheet_name=name, index=False)
        ws = xw.sheets[name]; ws.freeze_panes = 'A2'
        for i, col in enumerate(df.columns, 1):
            width = min(max(len(str(col)), int(df[col].astype(str).str.len().quantile(0.9)) if len(df) else 10) + 2, 45)
            ws.column_dimensions[ws.cell(1, i).column_letter].width = width
print('Saved', out, '-', len(sheets), 'sheets')
try:
    files.download(out)
except Exception:
    print('Download manually from the Files panel on the left.')

Saved Olist_Funnel_Excel_Data.xlsx - 18 sheets


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Key Insights

1. **Funnel conversion:** Olist received 8,000 Marketing Qualified Leads (MQLs) and 842 became closed deals, an overall conversion rate of 10.5% (about 1 in 10).
2. **Data coverage caveat (read before comparing months):** the first deal in the data was won in Dec 2017. Jun-Dec 2017 cohorts converted at 3.0% (61 of 2,002) versus 13.0% (781 of 5,998) for Jan-May 2018, and MQL volume jumped from 200 (Dec) to 1,141 (Jan). Early months are not comparable with later ones, so Jan 2018 onward is the better benchmark. The May 2018 cohort (9.98%) is below Jan-Apr (13.3%-14.5%) and should be tracked, since part of the gap may be follow-up time.
3. **Lead origin:** organic search and paid search bring 48.5% of MQLs and 55.3% of closed deals, with about 12% conversion each. Social brings 1,350 MQLs but converts at only 5.6%; email converts at 3.0%.
4. **Origin tracking gap:** 1,159 MQLs (14.5%) have an unknown or blank origin and 193 of them closed (16.7%). This is a tracking problem, not proof that these are the best leads.
5. **Sales cycle has a long tail:** median time to close is 14 days but the average is 48.5; 65.9% of deals close within 30 days and 17.8% take more than 90 days.
6. **Closed-deal profile (shares of won deals, not conversion rates):** online_medium leads are 39.4% of closed deals and the cat behaviour profile is 48.3% (21% have no profile). Wolf-profile deals take longer (median 36 days vs 13 for cat).
7. **Sales team workload:** the top five SDRs handled 49.9% of closed deals and the top five SRs 49.4%. Leads worked per person are not in the data, so this shows workload, not performance.

### Business Recommendations

1. Review lead quality and follow-up for **social and email**, which bring high volume but low conversion.
2. Keep **organic and paid search** as the core channels (largest share of closed deals at about 12% conversion). Always compare channels on volume and conversion together.
3. **Improve origin tracking** so the 14.5% of MQLs with unknown or blank origin can be attributed to a channel.
4. Use **Jan 2018 onward as the benchmark**, confirm with the data owners whether deal recording changed around Dec 2017, and compare cohorts on a fixed window (for example conversion within 30 days) so older and newer months are treated fairly.
5. Set a **follow-up checkpoint around 30 days** and review 90+ day deals separately (including slower behaviour profiles such as wolf).
6. **Capture lead type and segment at the MQL stage**, so true conversion rates by lead type and segment can be measured.
7. Review how closed deals are spread across SDRs and SRs to check whether the concentration comes from lead allocation, capacity or specialisation. Do not rank people without leads-worked data.